In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
from pyproj import Geod


# ============================================================
# 1. Configuration
# ============================================================

OUTPUTS = Path("../outputs")

HW = Path(
    "../outputs/heatwave"
)

CITY_STATIC = Path("../data/city_attributes.csv")

OUT_BASE = Path(
    "../outputs/final_panel_full"
)

SCHEMES = [
    "local_p90",
    "local_p95",
    "city_p90",
    "city_p95",
    "fixed_tx35_tn25",
]

K = ["city", "cell_id"]
KEY = K + ["year"]
YEARS = list(range(2016, 2024))
HEAT = ["day", "night", "compound"]

GEOD = Geod(ellps="WGS84")

CLIMATE = [
    "t2m_mean_C",
    "d2m_mean_C",
    "skt_mean_C",
    "sp_mean_hPa",
    "rh_mean_pct",
    "tp_annual_mm",
    "t2m_monthly_mean_max_C",
    "t2m_monthly_mean_min_C",
    "rh_monthly_proxy_max_pct",
]

ZVARS = [
    "bld_coverage",
    "bld_mean_h_m",
    "bld_std_h_m",
    "lcz_1",
    "lcz_2",
    "lcz_3",
    "ndvi_mean",
    "water_coverage",
    "isa_mean_prob",
]

ML_COLS = [
    v + "_z"
    for v in ZVARS
] + [
    "cdd_22C_city",
    "rh_mean_pct",
    "tp_annual_mm",
    "lat",
    "dem_mean_m",
    "coast_dist_km",
    "koppen_B",
    "koppen_C",
    "region_SA",
    "region_SEA",
    "region_LA",
    "year_num",
]

DML_COLS = [
    "bld_coverage_z",
    "bld_mean_h_m_z",
    "lat",
    "dem_mean_m",
    "dem_mean_slope",
    "coast_dist_km",
    "rh_mean_pct",
    "tp_annual_mm",
    "cdd_22C_city",
    "water_coverage_z",
]

EXTRA_COLS = [
    "ndvi_mean_z",
    "isa_mean_prob_z",
    "dist_to_cbd_km",
]

FILES = {
    "building":
        OUTPUTS / "building" / "building_smod_all_2016_2023_filled.csv",

    "green_water":
        OUTPUTS / "vegetation_water" / "green_water_smod_all_2016_2023_waterfilled.csv",

    "isa":
        OUTPUTS / "impervious" / "isa_smod_all_2016_2023_filled.csv",

    "lcz":
        OUTPUTS / "lcz" / "lcz_smod_2016_2023.csv",

    "terrain":
        OUTPUTS / "terrain" / "terrain_smod_2016_2023.csv",

    "cdd":
        OUTPUTS / "cdd" / "cdd_panel_2016_2023.csv",

    "coast":
        OUTPUTS / "coast_distance" / "coast_distance_652.csv",

    "cbd":
        OUTPUTS / "cbd_distance" / "cbd_distance_652.csv",

    "city_static": CITY_STATIC,
}


# ============================================================
# 2. Basic functions
# ============================================================

def require(ok, msg):
    if not bool(ok):
        raise RuntimeError(msg)


def read(path):
    return pd.read_csv(
        path,
        float_precision="round_trip",
    )


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1 << 20), b""
        ):
            h.update(chunk)

    return h.hexdigest()


def as_bool(s):
    v = (
        s.astype(str)
        .str.strip()
        .str.lower()
    )

    require(
        v.isin([
            "true", "false",
            "1", "0",
            "1.0", "0.0",
        ]).all(),
        f"Unknown boolean value: {s.name}",
    )

    return v.isin(["true", "1", "1.0"])


def validate(df, annual=True):
    keys = KEY if annual else K

    require(
        len(df) == (5216 if annual else 652)
        and not df.duplicated(keys).any(),
        "Abnormal record count / duplicate keys",
    )

    require(
        df.city.nunique() == 35
        and len(df[K].drop_duplicates()) == 652,
        "Abnormal number of cities/grid cells",
    )

    if annual:
        require(
            df.groupby(K).year.apply(
                lambda x: set(x) == set(YEARS)
            ).all(),
            "Incomplete years",
        )


def attach(
    p,
    other,
    label,
    public=(),
    prefixes=(),
    rename=None,
):
    d = other.rename(
        columns=rename or {}
    ).copy()

    annual = "year" in d

    if annual:
        d = d[d.year.isin(YEARS)]

    validate(d, annual)
    keys = KEY if annual else K

    require(
        set(map(tuple, d[keys].to_numpy()))
        == set(
            map(
                tuple,
                p[keys].drop_duplicates().to_numpy(),
            )
        ),
        f"{label} keys do not match",
    )

    if {"lat", "lon"}.issubset(d.columns):
        coords = d[
            K + ["lat", "lon"]
        ].drop_duplicates()

        require(
            len(coords) == 652
            and not coords.duplicated(K).any(),
            f"{label} coordinates are not unique",
        )

        chk = (
            p[K + ["lat", "lon"]]
            .drop_duplicates()
            .merge(
                coords,
                on=K,
                validate="one_to_one",
                suffixes=("", "_other"),
            )
        )

        require(
            np.allclose(
                chk[["lat", "lon"]],
                chk[["lat_other", "lon_other"]],
                rtol=0,
                atol=1e-5,
            ),
            f"{label} coordinates do not match",
        )

        d = d.drop(columns=["lat", "lon"])

    ren = {
        c: (
            c
            if c in public
            or c.startswith(tuple(prefixes))
            else label + "__" + c
        )
        for c in d
        if c not in keys
    }

    d = d.rename(columns=ren)

    require(
        not (set(p) & (set(d) - set(keys))),
        f"{label} column name conflict",
    )

    return p.merge(
        d,
        on=keys,
        how="left",
        validate=(
            "one_to_one"
            if annual
            else "many_to_one"
        ),
    )


# ============================================================
# 3. Fill missing climate with the nearest fully observed grid cell of the same city and year
# ============================================================

def nearest_fill(p, target, columns, label):
    for col in columns:
        require(
            col + "_observed" not in p,
            f"Original-value column already exists: {col}",
        )
        p[col + "_observed"] = p[col]

    donors_ok = np.isfinite(
        p[columns].to_numpy(float)
    ).all(axis=1)

    p[label + "_filled"] = False
    p[label + "_fill_method"] = "observed"

    p.loc[
        ~donors_ok,
        label + "_fill_method",
    ] = "missing_unfilled"

    for field in [
        "donor_cell",
        "donor_year",
        "donor_lat",
        "donor_lon",
        "dist_km",
    ]:
        p[label + "_fill_" + field] = np.nan

    logs = []

    for idx, row in p.loc[target].iterrows():
        donors = p[
            p.city.eq(row.city)
            & p.year.eq(row.year)
            & donors_ok
        ].sort_values("cell_id")

        require(
            len(donors) > 0,
            f"{row.city}/{row.year}/{label}"
            "no fully observed donor grid cell",
        )

        _, _, meters = GEOD.inv(
            np.full(len(donors), row.lon),
            np.full(len(donors), row.lat),
            donors.lon.to_numpy(),
            donors.lat.to_numpy(),
        )

        j = int(np.argmin(meters))
        donor = donors.iloc[j]

        p.loc[idx, columns] = (
            donor[columns].to_numpy(float)
        )

        p.at[idx, label + "_filled"] = True

        p.at[
            idx,
            label + "_fill_method",
        ] = "nearest_observed_cell_same_city_year_WGS84"

        values = {
            "donor_cell": int(donor.cell_id),
            "donor_year": int(donor.year),
            "donor_lat": float(donor.lat),
            "donor_lon": float(donor.lon),
            "dist_km": float(meters[j] / 1000),
        }

        for field, value in values.items():
            p.at[
                idx,
                label + "_fill_" + field,
            ] = value

        logs.append(
            dict(
                city=row.city,
                cell_id=int(row.cell_id),
                year=int(row.year),
                variable_group=label,
                **values,
            )
        )

    require(
        np.isfinite(
            p.loc[target, columns].to_numpy(float)
        ).all(),
        f"{label} still missing after filling",
    )

    return p, logs


# ============================================================
# 4. Merge common indicators
# ============================================================

def build_common():
    p = read(
        OUTPUTS / "panel_exposure_climate" / "panel_local_p90_2016_2023.csv"
    )

    validate(p)

    p = attach(
        p,
        read(FILES["building"]),
        "building",
        public=[
            "n_bld_pixels",
            "n_bld_pixels_observed",
            "gross_floor_area_m2",
            "gross_floor_area_m2_observed",
            "eligible_density_model",
            "eligible_height_model",
        ],
        prefixes=["bld_"],
        rename={
            "coverage_fraction": "bld_valid_frac",
            "coverage_pass": "bld_coverage_pass",
        },
    )

    p = attach(
        p,
        read(FILES["green_water"]),
        "green_water",
        prefixes=[
            "ndvi_", "water_", "veg_", "green_water_"
        ],
    )

    p = attach(
        p,
        read(FILES["isa"]),
        "isa",
        prefixes=["isa_"],
    )

    p = attach(
        p,
        read(FILES["lcz"]),
        "lcz",
        prefixes=["lcz_"],
    )

    p = attach(
        p,
        read(FILES["terrain"]),
        "terrain",
        prefixes=[
            "dem_", "elevation_", "slope_",
            "terrain_",
        ],
        rename={
            "elev_mean_minus_median_observed":
                "dem_mean_minus_median_m"
        },
    )

    p = attach(
        p,
        read(FILES["cdd"]),
        "cdd",
        prefixes=["cdd_"],
        rename={
            "n_cells_complete": "cdd_city_n_cells"
        },
    )

    p = attach(
        p,
        read(FILES["coast"]),
        "coast",
        prefixes=["coast_", "nearest_coast_"],
        rename={
            "needs_review": "coast_needs_review"
        },
    )

    p = attach(
        p,
        read(FILES["cbd"]),
        "cbd",
        prefixes=[
            "dist_to_",
            "nearest_cbd",
        ],
    )

    boolean_columns = [
        "heatwave_annual_complete",
        "baseline_eligible",
        "climate_main_complete",
        "climate_all_complete",
        "eligible_density_model",
        "eligible_height_model",
        "bld_filled",
        "water_filled_from_other_years",
        "isa_filled",
        "isa_imputed",
        "lcz_coverage_pass",
        "coast_needs_review",
    ]

    for col in boolean_columns:
        p[col] = as_bool(p[col])

    require(
        p.isa_imputed.equals(p.isa_filled),
        "ISA fill flag inconsistent",
    )

    require(
        p.lcz_coverage_pass.equals(
            p.lcz_coverage_fraction.ge(0.9)
        ),
        "Wrong LCZ coverage flag",
    )

    require(
        p.loc[
            ~p.lcz_coverage_pass,
            ["lcz_1", "lcz_2", "lcz_3"],
        ].isna().all().all(),
        "Final values of low-coverage LCZ not set to missing",
    )

    for col in [
        "climate_main_complete",
        "climate_all_complete",
        "heatwave_and_main_climate_complete",
        "climate_missing_variables",
        "climate_missing_reason",
        "climate_imputed",
    ]:
        p[col + "_observed"] = p[col]

    target = (
        p.heatwave_annual_complete
        & ~p.climate_main_complete
    )

    p, clogs = nearest_fill(
        p, target, CLIMATE, "climate"
    )

    p["climate_imputed"] = p.climate_filled

    p["climate_main_complete"] = np.isfinite(
        p[
            ["t2m_mean_C", "rh_mean_pct", "tp_annual_mm"]
        ].to_numpy(float)
    ).all(axis=1)

    p["climate_all_complete"] = np.isfinite(
        p[CLIMATE[:6]].to_numpy(float)
    ).all(axis=1)

    p["heatwave_and_main_climate_complete"] = (
        p.heatwave_annual_complete
        & p.climate_main_complete
    )

    p.loc[
        p.climate_filled,
        [
            "climate_missing_variables",
            "climate_missing_reason",
        ],
    ] = ""

    st = read(FILES["city_static"])[
        ["city", "region", "koppen_code", "koppen_broad"]
    ]

    require(
        st.notna().all().all()
        and st.groupby("city")
        .nunique(dropna=False)
        .eq(1).all().all(),
        "City static attributes are not unique / missing",
    )

    st = st.drop_duplicates("city")

    require(
        set(st.city) == set(p.city),
        "City list of static attributes differs",
    )

    p = p.merge(
        st,
        on="city",
        how="left",
        validate="many_to_one",
    )

    require(
        set(p.region) == {
            "Africa",
            "South Asia",
            "Southeast Asia",
            "Latin America",
        },
        "Abnormal region category",
    )

    require(
        set(p.koppen_broad) == {
            "Tropical (A)",
            "Arid (B)",
            "Temperate/Highland (C)",
        },
        "Abnormal Koppen category",
    )

    p["koppen_B"] = (
        p.koppen_broad.eq("Arid (B)").astype(int)
    )

    p["koppen_C"] = (
        p.koppen_broad
        .eq("Temperate/Highland (C)")
        .astype(int)
    )

    for label, col in [
        ("South Asia", "region_SA"),
        ("Southeast Asia", "region_SEA"),
        ("Latin America", "region_LA"),
    ]:
        p[col] = p.region.eq(label).astype(int)

    p["year_num"] = p.year - YEARS[0]

    for year in YEARS[1:]:
        p[f"yr_{year}"] = p.year.eq(year).astype(int)

    require(
        p.study_area_km2.gt(0).all()
        and p.pop_study.ge(0).all(),
        "Abnormal population / study area",
    )

    p["pop_total"] = p.pop_study

    p["pop_density_km2"] = (
        p.pop_study / p.study_area_km2
    )

    require(
        np.allclose(
            p.pop_density_km2,
            p.pop_study_density_km2,
            rtol=1e-10,
            atol=1e-8,
        ),
        "Population density does not match the population source table",
    )

    params = []

    for col in ZVARS:
        x = p[col]
        mu = float(x.mean())
        sd = float(x.std(ddof=1))

        require(
            np.isfinite(sd) and sd > 0,
            f"Cannot standardise: {col}",
        )

        p[col + "_z"] = (x - mu) / sd

        params.append(
            dict(
                variable=col,
                n=int(x.notna().sum()),
                mean=mu,
                sd=sd,
                ddof=1,
            )
        )

    return (
        p,
        pd.DataFrame(clogs),
        pd.DataFrame(params),
    )


# ============================================================
# 5. Add the results of each heatwave scheme and build separate sample flags
# ============================================================

def make_scheme(common, scheme):
    source = read(
        OUTPUTS / "panel_exposure_climate" / f"panel_{scheme}_2016_2023.csv"
    ).set_index(KEY)

    p = common.copy()

    dynamic = [
        "scheme",
        "annual_complete",
        "quality_status",
        "heatwave_annual_complete",
        "baseline_eligible",
    ]

    dynamic += [
        c for c in source
        if c.startswith(
            ("HW_days_", "HWDD_", "exposure_")
        )
    ]

    order = pd.MultiIndex.from_frame(p[KEY])

    require(
        set(source.index) == set(order),
        f"{scheme} keys do not match",
    )

    for col in dynamic:
        p[col] = source.loc[
            order, col
        ].to_numpy()

    p["heatwave_annual_complete"] = as_bool(
        p.heatwave_annual_complete
    )

    p["baseline_eligible"] = as_bool(
        p.baseline_eligible
    )

    p["heatwave_quality_status"] = p.quality_status

    p["pct"] = {
        "local_p90": "p90",
        "local_p95": "p95",
    }.get(scheme, scheme)

    hw = read(
        HW / f"annual_{scheme}_wide.csv"
    )

    hw = hw[hw.year.isin(YEARS)]
    validate(hw)

    h = hw.set_index(KEY).loc[order]

    for col in (
        ["HW_days_" + t for t in HEAT]
        + ["HWDD_" + t for t in HEAT]
    ):
        require(
            np.allclose(
                p[col],
                h[col].to_numpy(),
                equal_nan=True,
                rtol=1e-12,
                atol=1e-10,
            ),
            f"Heatwave source values do not match: {scheme}/{col}",
        )

    for t in HEAT:
        for metric in [
            "HW_events",
            "HW_max_duration",
            "HW_mean_intensity",
            "HW_max_intensity",
        ]:
            col = f"{metric}_{t}"
            p[col] = h[col].to_numpy()

        for unit, variable in [
            ("days", "HW_days"),
            ("Cdays", "HWDD"),
        ]:
            require(
                np.allclose(
                    p[f"exposure_people_{unit}_{t}"],
                    p.pop_study * p[f"{variable}_{t}"],
                    equal_nan=True,
                    rtol=1e-9,
                    atol=1e-6,
                ),
                f"Exposure product does not match: {scheme}/{t}/{unit}",
            )

        require(
            p[f"exposure_people_days_{t}"]
            .dropna().ge(0).all(),
            "Negative exposure",
        )

        p[f"log_exp_{t}"] = np.log1p(
            p[f"exposure_people_days_{t}"]
        )

        p[f"percap_hwd_{t}"] = p[f"HW_days_{t}"]

    p["heatwave_and_main_climate_complete"] = (
        p.heatwave_annual_complete
        & p.climate_main_complete
    )

    baseok = (
        p.heatwave_annual_complete
        & p.baseline_eligible
        & p.eligible_density_model
        & p.eligible_height_model
    )

    p["sample_ml_controls"] = (
        baseok
        & np.isfinite(
            p[ML_COLS].to_numpy(float)
        ).all(axis=1)
    )

    p["sample_dml_controls"] = (
        baseok
        & np.isfinite(
            p[DML_COLS].to_numpy(float)
        ).all(axis=1)
    )

    p["sample_dml_context_controls"] = (
        p.sample_dml_controls
        & np.isfinite(
            p[EXTRA_COLS].to_numpy(float)
        ).all(axis=1)
    )

    p["any_main_input_imputed"] = (
        p.bld_filled
        | p.water_filled_from_other_years
        | p.isa_imputed
        | p.climate_imputed
    )

    for t in HEAT:
        p[f"sample_ml_{t}"] = (
            p.sample_ml_controls
            & p[f"log_exp_{t}"].notna()
        )

        p[f"sample_ml_{t}_observed_only"] = (
            p[f"sample_ml_{t}"]
            & ~p.any_main_input_imputed
        )

        p[f"sample_dml_{t}"] = (
            p.sample_dml_controls
            & p[f"HW_days_{t}"].notna()
            & p.pop_study.gt(0)
        )

        p[
            f"sample_dml_{t}_positive_log"
        ] = (
            p[f"sample_dml_{t}"]
            & p[f"HW_days_{t}"].gt(0)
            & p[f"exposure_people_days_{t}"].gt(0)
        )

    validate(p)

    return (
        p.sort_values(KEY)
        .reset_index(drop=True)
    )


# ============================================================
# 6. Save the five complete panels and the validation report
# ============================================================

def run_final():
    files = dict(FILES)

    for scheme in SCHEMES:
        files["base_" + scheme] = (
            OUTPUTS / "panel_exposure_climate" / f"panel_{scheme}_2016_2023.csv"
        )

        files["heatwave_" + scheme] = (
            HW / f"annual_{scheme}_wide.csv"
        )

    for label, path in files.items():
        require(
            path.is_file(),
            f"Missing {label}: {path}",
        )

    manifest = [
        dict(
            name=k,
            path=str(v),
            sha256=sha256(v),
        )
        for k, v in files.items()
    ]

    out = OUT_BASE

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    report = dict(
        status="running",
        rows_per_scheme=5216,
        points=652,
        years=YEARS,
    )

    def write_report():
        (out / "summary.json").write_text(
            json.dumps(
                report,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    def save(df, name):
        df = df.reset_index(drop=True)

        df.to_csv(
            out / name,
            index=False,
            encoding="utf-8-sig",
            float_format="%.17g",
        )

        back = read(out / name).replace(
            {"": np.nan}
        )

        expected = df.replace(
            {"": np.nan}
        )

        pd.testing.assert_frame_equal(
            expected,
            back,
            check_dtype=False,
            check_exact=True,
        )

    write_report()
    print("Output folder:", out, flush=True)

    try:
        common, fill_log, zparams = build_common()

        panels = {}
        counts = []

        for scheme in SCHEMES:
            p = make_scheme(common, scheme)
            panels[scheme] = p

            save(
                p,
                f"panel_final_{scheme}_2016_2023.csv",
            )

            counts.append(
                dict(
                    scheme=scheme,
                    rows=len(p),
                    climate_filled=int(
                        p.climate_filled.sum()
                    ),
                    ml_controls=int(
                        p.sample_ml_controls.sum()
                    ),
                    dml_controls=int(
                        p.sample_dml_controls.sum()
                    ),
                    ml_observed_only=int(
                        (
                            p.sample_ml_controls
                            & ~p.any_main_input_imputed
                        ).sum()
                    ),
                )
            )

        main = panels["local_p90"].copy()

        p95 = (
            panels["local_p95"]
            .set_index(KEY)
            .loc[
                pd.MultiIndex.from_frame(main[KEY])
            ]
        )

        for t in HEAT:
            for col in [
                f"HW_days_{t}",
                f"exposure_people_days_{t}",
                f"log_exp_{t}",
            ]:
                main[col + "_p95"] = (
                    p95[col].to_numpy()
                )

        save(
            main,
            "panel_final_main_local_p90_with_p95_2016_2023.csv",
        )

        save(
            fill_log,
            "climate_fill_sources.csv",
        )

        save(
            zparams,
            "standardization_parameters.csv",
        )

        save(
            pd.DataFrame(counts),
            "model_availability_by_scheme.csv",
        )

        save(
            pd.DataFrame(manifest),
            "input_manifest.csv",
        )

        save(
            main[
                KEY + [
                    "lat",
                    "lon",
                    "coast_needs_review",
                    "pop_nodata_share_study",
                    "lcz_coverage_fraction",
                    "lcz_coverage_pass",
                    "bld_valid_frac",
                    "bld_coverage_pass",
                    "bld_filled",
                    "isa_imputed",
                    "water_filled_from_other_years",
                    "climate_imputed",
                    "sample_ml_controls",
                    "sample_dml_controls",
                ]
            ],
            "quality_and_sample_flags.csv",
        )

        missing = main.isna().sum()

        save(
            missing.rename_axis("column")
            .reset_index(name="missing_rows"),
            "missing_values_main.csv",
        )

        for row in manifest:
            require(
                sha256(row["path"]) == row["sha256"],
                f"Source file changed during the run: {row['path']}",
            )

        report.update(
            status="completed",
            schemes=counts,
            main_file=(
                "panel_final_main_local_p90_with_p95_2016_2023.csv"
            ),
            readback_verified_all_outputs=True,
            columns=len(main.columns),
            climate_max_donor_distance_km=float(
                fill_log.loc[
                    fill_log.variable_group.eq("climate"),
                    "dist_km",
                ].max()
            ),
        )

        write_report()

    except (Exception, KeyboardInterrupt) as exc:
        report.update(
            status="failed_or_interrupted",
            error=repr(exc),
        )
        write_report()
        raise

    print("Output folder:", out)

    return out, main, panels


FINAL_PANEL_DIR, FINAL_MAIN, FINAL_PANELS = run_final()

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd

INPUT_RUN = Path("../outputs/final_panel_full")
OUT_BASE = Path("../outputs/final_panel")
KEY = ["city", "cell_id", "year"]
EXPECTED_ROWS = 5216
COMPANION_FILES = ["standardization_parameters.csv", "climate_fill_sources.csv",
                   "input_manifest.csv", "model_availability_by_scheme.csv", "summary.json"]

KEEP_ANALYSIS = [
    "city", "cell_id", "year", "scheme", "pct", "lat", "lon",
    "region", "koppen_code", "koppen_broad", "koppen_B", "koppen_C", "region_SA", "region_SEA", "region_LA",
    "year_num", "yr_2017", "yr_2018", "yr_2019", "yr_2020", "yr_2021", "yr_2022", "yr_2023",
    "heatwave_annual_complete", "eligible_density_model", "eligible_height_model",
    "sample_dml_controls", "sample_dml_context_controls", "sample_ml_controls",
    "sample_dml_day_positive_log", "sample_dml_night_positive_log",
    "sample_dml_compound_positive_log", "sample_ml_day_observed_only", "sample_ml_night_observed_only",
    "sample_ml_compound_observed_only", "any_main_input_imputed",
    "pop_study", "pop_total", "study_area_km2", "study_area_fraction", "pop_density_km2",
    "HW_days_day", "HWDD_day", "HW_events_day", "HW_max_duration_day", "HW_mean_intensity_day",
    "HW_max_intensity_day",
    "HW_days_night", "HWDD_night", "HW_events_night", "HW_max_duration_night", "HW_mean_intensity_night",
    "HW_max_intensity_night",
    "HW_days_compound", "HWDD_compound", "HW_events_compound", "HW_max_duration_compound",
    "HW_mean_intensity_compound", "HW_max_intensity_compound",
    "exposure_people_days_day", "exposure_people_days_night", "exposure_people_days_compound",
    "log_exp_day", "log_exp_night", "log_exp_compound",
    "exposure_people_Cdays_day", "exposure_people_Cdays_night", "exposure_people_Cdays_compound",
    "HW_days_day_p95", "exposure_people_days_day_p95", "log_exp_day_p95",
    "HW_days_night_p95", "exposure_people_days_night_p95", "log_exp_night_p95",
    "HW_days_compound_p95", "exposure_people_days_compound_p95", "log_exp_compound_p95",
    "bld_coverage", "bld_mean_h_m", "bld_std_h_m", "bld_coverage_z", "bld_mean_h_m_z", "bld_std_h_m_z",
    "bld_median_h_m", "bld_p90_h_m", "bld_max_h_m", "bld_IQR_m", "bld_CV", "bld_var_h_m2",
    "bld_low_share", "bld_mid_share", "bld_highrise_share", "bld_FAR", "bld_volume_density_m",
    "bld_footprint_m2", "gross_floor_area_m2", "bld_volume_m3", "n_bld_pixels",
    "rh_mean_pct", "tp_annual_mm", "t2m_mean_C", "cdd_22C_city", "cdd_22C",
    "dem_mean_m", "dem_mean_slope", "dem_std_m", "coast_dist_km", "dist_to_cbd_km",
    "water_coverage", "water_coverage_z", "ndvi_mean", "ndvi_mean_z", "veg_coverage",
    "isa_mean_prob", "isa_mean_prob_z",
    "lcz_1", "lcz_2", "lcz_3", "lcz_4", "lcz_5", "lcz_6", "lcz_7", "lcz_8", "lcz_9", "lcz_10", "lcz_11",
    "lcz_12", "lcz_13", "lcz_14", "lcz_15", "lcz_16", "lcz_17", "lcz_urban_frac", "lcz_natural_frac",
    "lcz_dominant", "lcz_n_classes", "lcz_1_z", "lcz_2_z", "lcz_3_z",
]
KEEP_QA = [
    "heatwave_quality_status", "pop_cell_total", "pop_nodata_share_study", "pop_nodata_share_cell",
    "pop_study_density_km2", "pop_study_share",
    "building__source_kind", "bld_valid_frac", "bld_coverage_pass", "bld_filled", "bld_fill_source_year",
    "ndvi_valid_frac", "ndvi_coverage_pass", "water_valid_frac", "water_coverage_pass",
    "water_filled_from_other_years", "water_fill_donor_years",
    "isa_valid_frac", "isa_coverage_pass", "isa_filled", "isa_fill_method", "isa_fill_gap_years",
    "lcz_coverage_fraction", "lcz_coverage_pass",
    "climate_main_complete", "climate_filled", "climate_fill_donor_cell", "climate_fill_dist_km",
    "cdd_city_n_cells",
    "dem_mean_minus_median_m", "coast_needs_review", "nearest_cbd",
]
P95 = [c for c in KEEP_ANALYSIS if c.endswith("_p95")]
MUST_KEEP = ["pct", "pop_total", "heatwave_annual_complete", "sample_dml_controls"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def same_values(a, b):
    for c in a.columns:
        x, y = a[c], b[c]
        if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
            ok = (x.to_numpy() == y.to_numpy()) | (x.isna().to_numpy() & y.isna().to_numpy())
        else:
            ok = (x.astype(str).to_numpy() == y.astype(str).to_numpy())
        if not ok.all():
            return c
    return None


def run_select():
    require(len(set(KEEP_ANALYSIS)) == len(KEEP_ANALYSIS) and len(set(KEEP_QA)) == len(KEEP_QA)
            and not set(KEEP_ANALYSIS) & set(KEEP_QA), "duplicates in the keep lists")
    files = sorted(INPUT_RUN.glob("panel_final_*_2016_2023.csv"))
    require(len(files) > 0, f"{INPUT_RUN}: panel_final_*_2016_2023.csv not found")
    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)
    print("Output folder:", out_dir, flush=True)

    report = []
    for f in files:
        df = pd.read_csv(f, low_memory=False, float_precision="round_trip")
        require(len(df) == EXPECTED_ROWS and not df.duplicated(KEY).any(), f"{f.name}: abnormal row count or keys")
        is_main = "_with_p95_" in f.name
        want = [c for c in KEEP_ANALYSIS if is_main or c not in P95]
        miss = [c for c in want + KEEP_QA if c not in df.columns]
        require(not miss, f"{f.name} missing columns: {miss}")
        require(all(c in want for c in MUST_KEEP), "columns required by DML are not in the keep list")

        analysis_cols = want
        ana = df[analysis_cols]
        dropped = [c for c in df.columns if c not in analysis_cols and c not in KEEP_QA]
        out_f = out_dir / f.name
        ana.to_csv(out_f, index=False, encoding="utf-8-sig", float_format="%.17g")
        back = pd.read_csv(out_f, low_memory=False, float_precision="round_trip")
        require(back.shape == ana.shape and list(back.columns) == analysis_cols, f"{out_f.name}: columns read back do not match")
        bad = same_values(back, df[analysis_cols])
        require(bad is None, f"{out_f.name}: values of column {bad} changed after writing")

        row = dict(file=f.name, rows=len(df), columns_in=df.shape[1], columns_analysis=len(analysis_cols),
                   columns_dropped=len(dropped))
        qa = df[KEY + KEEP_QA]
        qa_f = out_dir / f.name.replace(".csv", "_qa.csv")
        qa.to_csv(qa_f, index=False, encoding="utf-8-sig", float_format="%.17g")
        qback = pd.read_csv(qa_f, low_memory=False, float_precision="round_trip")
        require(qback.shape == qa.shape and same_values(qback, qa) is None, f"{qa_f.name}: read-back mismatch")
        row["columns_qa_file"] = len(KEY + KEEP_QA)
        report.append(row)
        print(f"{f.name}: {df.shape[1]} columns → analysis {len(analysis_cols)} columns"
              + f" + QA {len(KEEP_QA)} columns (saved separately)"
              + f", dropped {len(dropped)} columns", flush=True)

    for name in COMPANION_FILES:
        if (INPUT_RUN / name).is_file():
            shutil.copy2(INPUT_RUN / name, out_dir / f"source_{name}")
    pd.DataFrame(report).to_csv(out_dir / "selection_report.csv", index=False, encoding="utf-8-sig")
    summary = dict(status="completed", input_run=str(INPUT_RUN), keep_analysis=KEEP_ANALYSIS, keep_qa=KEEP_QA, files=report)
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Output folder:", out_dir)
    return out_dir


SELECTED_DIR = run_select()